<a href="https://colab.research.google.com/github/Sparten-Ashvinee/IISc-LLM/blob/main/Assignments/Inference_profiling_and_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yaml
import torch
import matplotlib.pyplot as plt
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer

In [2]:
class GPT2_Model:
    def __init__(self, config):
        self.config = config

    def model(self):
        self.configuration = GPT2Config(**self.config)
        self.configuration._attn_implementation = "eager"
        self.model_ = GPT2LMHeadModel(self.configuration)

        return self.model_

    def print_param(self, model):

        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total_params:,}")

        fp32_bytes = total_params * 4
        fp16_bytes = total_params * 2

        print(f"FP32 memory: {fp32_bytes / 1024**2:.2f} MB")
        print(f"FP16 memory: {fp16_bytes / 1024**2:.2f} MB")

        param_memory = fp32_bytes/1024**2
        return param_memory

In [3]:
def get_register_hook(module):
    def get_forward_hook(name):
        def forward_hook_fun(module, inp, out):
            if isinstance(module, torch.nn.Embedding):
                x = inp[0]
                print('Embedding: ',name,'has shape: ',out.shape)
            if 'c_attn' in name:
                print('Attention: ',name,'has shape: ',out.reshape((out.shape[0],3,out.shape[1],out.shape[2]//3)).shape)
            if 'act' in name:
                print("Activation dim:", out.shape)
            if 'ln_f' in name:
                print("LayerNorm dim:", out.shape)
        return forward_hook_fun

    for name, layer in module.named_modules():
        layer.register_forward_hook(get_forward_hook(name))

In [19]:
data_path = '/content/data.txt'
with open(data_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")

# Use GPU if available, else fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
tokenizer("Hello sir Hello sir", return_tensors="pt").to(device)

{'input_ids': tensor([[15496, 15967, 18435, 15967]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1]], device='cuda:0')}

In [21]:
#GPT2 Small Model
print("\nGPT2 Small:")
with open("/content/gpt2_large.yaml", "r") as f:
    config_small = yaml.safe_load(f)

#print(config_small)
gpt2_small = GPT2_Model(config_small)
gpt2_small_model = gpt2_small.model()
gpt2_small_model = gpt2_small_model.to(device)

inputs = tokenizer(raw_text, return_tensors="pt").to(device)

get_register_hook(gpt2_small_model)

input_test = inputs.input_ids[0][:1024]
#print(input_test)
with torch.no_grad():
    outputs = gpt2_small_model(input_test.unsqueeze(0), labels=input_test.unsqueeze(0), output_attentions=True)

print("Attention scores dim:", outputs.attentions[0].shape)

small_act_memory=outputs.attentions[0].shape[0] * outputs.attentions[0].shape[1] * outputs.attentions[0].shape[2] * outputs.attentions[0].shape[3] * 4/1024**2

#small_act_memory=max(act_memory)
print("Peak activation memory:",small_act_memory, "MB")

small_param_memory = gpt2_small.print_param(gpt2_small_model)

# Free Small model before loading Medium
del gpt2_small_model, gpt2_small, outputs, inputs
torch.cuda.empty_cache()


GPT2 Small:
{'activation_function': 'gelu_new', 'add_cross_attention': False, 'attn_pdrop': 0.1, 'bos_token_id': 50256, 'embd_pdrop': 0.1, 'eos_token_id': 50256, 'initializer_range': 0.02, 'layer_norm_epsilon': 1e-05, 'model_type': 'gpt2', 'n_embd': 1280, 'n_head': 20, 'n_inner': None, 'n_layer': 36, 'n_positions': 1280, 'pad_token_id': None, 'reorder_and_upcast_attn': False, 'resid_pdrop': 0.1, 'scale_attn_by_inverse_layer_idx': False, 'scale_attn_weights': True, 'summary_activation': None, 'summary_first_dropout': 0.0, 'summary_proj_to_labels': True, 'summary_type': 'cls_index', 'summary_use_proj': True, 'tie_word_embeddings': True, 'transformers_version': '5.2.0', 'use_cache': True, 'vocab_size': 50257, 'attn_implementation': 'eager'}


Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


tensor([   40,   367,  2885,  ...,   691, 12226,   318], device='cuda:0')
Embedding:  transformer.wte has shape:  torch.Size([1, 1024, 1280])
Embedding:  transformer.wpe has shape:  torch.Size([1, 1024, 1280])
Attention:  transformer.h.0.attn.c_attn has shape:  torch.Size([1, 3, 1024, 1280])
Activation dim: torch.Size([1, 1024, 5120])
Attention:  transformer.h.1.attn.c_attn has shape:  torch.Size([1, 3, 1024, 1280])
Activation dim: torch.Size([1, 1024, 5120])
Attention:  transformer.h.2.attn.c_attn has shape:  torch.Size([1, 3, 1024, 1280])
Activation dim: torch.Size([1, 1024, 5120])
Attention:  transformer.h.3.attn.c_attn has shape:  torch.Size([1, 3, 1024, 1280])
Activation dim: torch.Size([1, 1024, 5120])
Attention:  transformer.h.4.attn.c_attn has shape:  torch.Size([1, 3, 1024, 1280])
Activation dim: torch.Size([1, 1024, 5120])
Attention:  transformer.h.5.attn.c_attn has shape:  torch.Size([1, 3, 1024, 1280])
Activation dim: torch.Size([1, 1024, 5120])
Attention:  transformer.h.6.

OutOfMemoryError: CUDA out of memory. Tried to allocate 80.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 41.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.11 GiB is allocated by PyTorch, and 288.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:

#GPT2 Medium Model
print("\nGPT2 Medium:")
with open("gpt2_medium.yaml", "r") as f:
    config_medium = yaml.safe_load(f)

gpt2_medium = GPT2_Model(config_medium)
gpt2_medium_model = gpt2_medium.model().to(device)

inputs = tokenizer(raw_text, return_tensors="pt").to(device)

get_register_hook(gpt2_medium_model)

input_test = inputs.input_ids[0][:1024]
with torch.no_grad():
    outputs = gpt2_medium_model(input_test.unsqueeze(0), labels=input_test.unsqueeze(0), output_attentions=True)

print("Attention scores dim:", outputs.attentions[0].shape)

# act_memory=[]
# for idx in range(len(outputs.attentions)):
#     mem = outputs.attentions[idx].shape[0] * outputs.attentions[idx].shape[1] * outputs.attentions[idx].shape[2] * outputs.attentions[idx].shape[3] * 4/1024**2
#     print("Memory Bottleneck", mem, "MB")
#     act_memory.append(mem)

#medium_act_memory=max(act_memory)

medium_act_memory=outputs.attentions[0].shape[0] * outputs.attentions[0].shape[1] * outputs.attentions[0].shape[2] * outputs.attentions[0].shape[3] * 4/1024**2
print("Peak activation memory:",medium_act_memory, "MB")

medium_param_memory = gpt2_medium.print_param(gpt2_medium_model)

# Free Medium model before loading Large
del gpt2_medium_model, gpt2_medium, outputs, inputs
torch.cuda.empty_cache()

In [ ]:
#GPT2 Large Model
print("\nGPT2 Large:")
with open("gpt2_large.yaml", "r") as f:
    config_large = yaml.safe_load(f)

gpt2_large = GPT2_Model(config_large)
gpt2_large_model = gpt2_large.model().to(device)

inputs = tokenizer(raw_text, return_tensors="pt").to(device)

get_register_hook(gpt2_large_model)

input_test = inputs.input_ids[0][:1024]
with torch.no_grad():
    outputs = gpt2_large_model(input_test.unsqueeze(0), labels=input_test.unsqueeze(0), output_attentions=True)

print("Attention scores dim:", outputs.attentions[0].shape)

# act_memory=[]
# for idx in range(len(outputs.attentions)):
#     mem = outputs.attentions[idx].shape[0] * outputs.attentions[idx].shape[1] * outputs.attentions[idx].shape[2] * outputs.attentions[idx].shape[3] * 4/1024**2
#     print("Memory Bottleneck", mem, "MB")
#     act_memory.append(mem)

#large_act_memory=max(act_memory)
large_act_memory=outputs.attentions[0].shape[0] * outputs.attentions[0].shape[1] * outputs.attentions[0].shape[2] * outputs.attentions[0].shape[3] * 4/1024**2
print("Peak activation memory:",large_act_memory, "MB")

large_param_memory = gpt2_large.print_param(gpt2_large_model)

In [ ]:
print("Ratio act/param memory for GPT2 Small:", small_act_memory/small_param_memory)
print("Ratio act/param memory for GPT2 Medium:", medium_act_memory/medium_param_memory)
print("Ratio act/param memory for GPT2 Large:", large_act_memory/large_param_memory)

In [ ]:
if small_act_memory/small_param_memory > medium_act_memory/medium_param_memory and small_act_memory/small_param_memory > large_act_memory/large_param_memory:
    print("GPT2 Small has negligible activation")
elif medium_act_memory/medium_param_memory > small_act_memory/small_param_memory and medium_act_memory/medium_param_memory > large_act_memory/large_param_memory:
    print("GPT2 Medium has negligible activation")
else:
    print("GPT2 Large has negligible activation")

In [ ]:
param_counts = [small_param_memory, medium_param_memory, large_param_memory]
fp16_memories = [small_param_memory, medium_param_memory, large_param_memory]
plt.figure(figsize=(8, 6))
plt.loglog(param_counts, fp16_memories, marker='o')
plt.title('Parameters Count vs FP16 Memory (Log-Log Scale)')
plt.xlabel('Parameters Count (MB)')
plt.ylabel('FP16 Memory (MB)')
plt.grid(True, which="both", ls="--")
plt.show()


print('Completed!')